# Browse IBD pairs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import socket as socket
import os as os
import sys as sys
import h5py
import matplotlib.cm as cm
import itertools as it
import multiprocessing as mp

socket_name = socket.gethostname()
print(socket_name)

if socket_name.startswith("bionc"):
    print("Leipzig Cluster detected!")
    path = "/r1/people/xiaowen_jia/anaconda3/lib/python3.9/site-packages/"
    #sys.path.append("/mnt/archgen/users/hringbauer/git/hapBLOCK/python3/")
    sys.path.insert(0, "/r1/people/xiaowen_jia/anaconda3/lib/python3.9/site-packages/ancIBD/")
else: 
    raise RuntimeWarning("Not compatible machine. Check!!")

os.chdir(path)  # Set the right Path (in line with Atom default)
print(os.getcwd())
print(f"CPU Count: {mp.cpu_count()}")

from ancIBD.plot.plot_df import plot_scatter_ibd, plot_sactter_ibd_target
from ancIBD.plot.plot_karyotype import plot_karyo_from_ibd_df
from ancIBD.ibd_stats.funcs import new_columns, find_relatives, give_sub_df, plot_age_diff, rc_date
#from ancIBD.ibd_stats.funcs import new_columns, find_relatives, give_sub_df, plot_age_diff, rc_date, remove_iids
from ancIBD.run import run_plot_pair

In [ ]:
v0 = "12.0" # The IBD version Number
meta_v = "0.5"
folder_out = f"/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/"

df_meta = pd.read_csv(f"/mnt/archgen/users/hringbauer/git/auto_popgen/data/iid_list/iid.ibd.meta.v12.0.unique.tsv",  sep="\t")
print(f"Loaded {len(df_meta)} IIDs from IBD run with meta info.")
#df_meta_pd = pd.read_csv("./git/auto_popgen/data/meta/pandora_meta.v1.tsv", sep="\t")

df_ibds = pd.read_csv(f"{folder_out}ibd220f.ind.{v0}.tsv",  sep="\t")
print(f"Loaded {len(df_ibds)} IID pairs")
#df_ibds = remove_iids(df_ibds, iids=["TML002"])


df_ibds = new_columns(df_ibds, df_meta, col=["frac_gp"]) # "Locality", "Province", "C14_mean", 
df_ibds = new_columns(df_ibds, df_meta, col=["Country", "date"]) # "Locality", "Province", "C14_mean", 

x = 0.7
df_ibds2 = df_ibds[(df_ibds["frac_gp1"]>x) & (df_ibds["frac_gp2"]>x)]
print(f"Loaded {len(df_ibds2)} IID pairs with GP_max frac > {x}")
df_ibds1 = pd.read_csv(f"{folder_out}ibd220f.ind.{v0}.tsv",  sep="\t")
df_ibds2.to_csv(f"{folder_out}iid220f.ind.{v0}.frac0.7.csv")

# Relatives across different sites

In [ ]:
import pandas as pd
import numpy as np

# --- Load ---
pairs = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv",
    sep=","
)

meta = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC.v12.iid.meta.csv"
)

# --- Period annotation (only need iid + period) ---
ann = meta[["iid", "period"]].copy()

# Attach periods to both ends
pairs = pairs.merge(
    ann.rename(columns={"iid": "iid1", "period": "period1"}),
    on="iid1", how="left"
).merge(
    ann.rename(columns={"iid": "iid2", "period": "period2"}),
    on="iid2", how="left"
)

# Keep only pairs where both periods are known
pairs = pairs.dropna(subset=["period1", "period2"]).copy()

# Optional: keep only these periods (comment out if you want all)
keep_periods = {"LBA", "EIA", "DIA"}
pairs = pairs[pairs["period1"].isin(keep_periods) & pairs["period2"].isin(keep_periods)].copy()

# --- Make pairs undirected + unique (avoid double-counting if present) ---
# Sort iids so (A,B) == (B,A)
iid_lo = np.minimum(pairs["iid1"].astype(str), pairs["iid2"].astype(str))
iid_hi = np.maximum(pairs["iid1"].astype(str), pairs["iid2"].astype(str))
pairs["pair_key"] = iid_lo + "||" + iid_hi

# Sort periods so (EIA,LBA) == (LBA,EIA)
p_lo = np.minimum(pairs["period1"].astype(str), pairs["period2"].astype(str))
p_hi = np.maximum(pairs["period1"].astype(str), pairs["period2"].astype(str))
pairs["period_pair"] = p_lo + "-" + p_hi

# Drop duplicate rows for the same undirected pair (if your file can contain duplicates)
pairs_u = pairs.drop_duplicates(subset=["pair_key", "period_pair"]).copy()

# --- Count the three requested period pairs ---
wanted = {"EIA-LBA", "DIA-EIA", "DIA-LBA"}  # after sorting, these are the canonical strings
counts = (
    pairs_u[pairs_u["period_pair"].isin(wanted)]
    .groupby("period_pair")
    .size()
    .reindex(sorted(wanted), fill_value=0)
)

print("Cross-period IBD link counts (unique undirected pairs):")
for k, v in counts.items():
    print(f"{k}: {v}")

# If you want them in your preferred order/names:
print("\nIn requested order:")
print(f"LBA–EIA: {int(counts.get('EIA-LBA', 0))}")
print(f"EIA–DIA: {int(counts.get('DIA-EIA', 0))}")
print(f"LBA–DIA: {int(counts.get('DIA-LBA', 0))}")


In [ ]:
import pandas as pd

# read the IBD files
df_ibds1 = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12.csv",
    sep=","
)

df_ibds2 = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_12cM_fracgp0.7.csv",
    sep=","
)

# read the TSV with iid and cts
df_iid = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/ibd_cts_phase3_from2.tsv",
    sep="\t"
)

# get the allowed iids as a set (fast lookup)
valid_iids = set(df_iid["iid"])

# filter: keep rows where BOTH iid1 and iid2 are in the iid list
df_ibds2 = df_ibds2[
    df_ibds2["iid1"].isin(valid_iids) &
    df_ibds2["iid2"].isin(valid_iids)
].sort_values(by="max_IBD", ascending=False)
df_ibds2.to_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_EIAonly_12cM_fracgp0.7.csv")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# --- Load ---
df_ibds2 = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv",
    sep=","
)

meta = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC.v12.iid.meta.csv")
df_iid = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/ibd_cts_phase3_from2.tsv",
    sep="\t"
)

# --- Prepare annotations: period + H/L ---
meta = meta[["iid", "period"]].copy()
df_iid = df_iid[["iid", "cts"]].copy()
df_iid["cts"] = pd.to_numeric(df_iid["cts"], errors="coerce")
df_iid["cts_group"] = np.where(df_iid["cts"] >= 10, "H", "L")

ann = meta.merge(df_iid[["iid", "cts_group"]], on="iid", how="left")

# --- Attach to pairs ---
pairs = df_ibds2.copy()

pairs = pairs.merge(
    ann.rename(columns={"iid": "iid1", "period": "period1", "cts_group": "group1"}),
    on="iid1", how="left"
)
pairs = pairs.merge(
    ann.rename(columns={"iid": "iid2", "period": "period2", "cts_group": "group2"}),
    on="iid2", how="left"
)

# --- EIA-LBA links either direction ---
eia_lba = pairs[
    ((pairs["period1"] == "EIA") & (pairs["period2"] == "LBA")) |
    ((pairs["period1"] == "LBA") & (pairs["period2"] == "EIA"))
].copy()

# Identify the EIA individual and its H/L group
eia_lba["EIA_iid"] = np.where(eia_lba["period1"] == "EIA", eia_lba["iid1"], eia_lba["iid2"])
eia_lba["EIA_group"] = np.where(eia_lba["period1"] == "EIA", eia_lba["group1"], eia_lba["group2"])

# Keep only EIA individuals with known H/L group
eia_lba = eia_lba.dropna(subset=["EIA_group"])

# --- Count EIA->LBA links per EIA individual (only those with >=1 link so far) ---
link_counts = eia_lba.groupby("EIA_iid").size()

# --- IMPORTANT: build the full denominator (all EIA individuals in H/L), then fill zeros ---
eia_all = ann[(ann["period"] == "EIA") & (ann["cts_group"].isin(["H", "L"]))].copy()

eia_all["n_LBA_links"] = eia_all["iid"].map(link_counts).fillna(0).astype(int)

# --- Descriptives using the FULL H/L sizes (including zeros) ---
print(
    eia_all.groupby("cts_group")["n_LBA_links"].agg(
        n="count", mean="mean", median="median", max="max",
        frac_any=lambda s: (s > 0).mean()
    )
)

# --- Test: do H have more links than L? (including zeros) ---
H = eia_all.loc[eia_all["cts_group"] == "H", "n_LBA_links"]
L = eia_all.loc[eia_all["cts_group"] == "L", "n_LBA_links"]

print("Mann–Whitney (H > L):", mannwhitneyu(H, L, alternative="greater"))


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal

# -----------------------------
# Inputs assumed to exist:
#   df_iid   with columns: ["iid", "cts"]
#   df_ibds2 with columns: ["iid1", "iid2", "max_IBD"]  (or change metric below)
# -----------------------------

IBD_METRIC = "max_IBD"   # change to "sum_IBD" etc. if you prefer

# --- 1) Bin cts into categories (2 groups: low/high). Use q=3 for tertiles, etc.
df_iid = df_iid.copy()
df_iid["cts"] = pd.to_numeric(df_iid["cts"], errors="coerce")
df_iid = df_iid.dropna(subset=["cts", "iid"])

df_iid["cts_group"] = pd.Series(
    np.where(df_iid["cts"] >= 15, "high", "low"),
    index=df_iid.index
)
df_iid.loc[df_iid["cts"].isna(), "cts_group"] = np.nan


# lookup maps
cts_group_map = df_iid.set_index("iid")["cts_group"]
cts_map = df_iid.set_index("iid")["cts"]

# --- 2) Attach group labels (and cts if you want) to each pair in df_ibds2
df_pairs = df_ibds2.copy()
df_pairs["g1"] = df_pairs["iid1"].map(cts_group_map)
df_pairs["g2"] = df_pairs["iid2"].map(cts_group_map)
df_pairs["cts1"] = df_pairs["iid1"].map(cts_map)
df_pairs["cts2"] = df_pairs["iid2"].map(cts_map)

# keep only pairs where both sides have a group label and metric is present
df_pairs[IBD_METRIC] = pd.to_numeric(df_pairs[IBD_METRIC], errors="coerce")
df_pairs = df_pairs.dropna(subset=["g1", "g2", IBD_METRIC])

# --- 3) Define pair types: HH, HL, LL
# (order-invariant label for mixed pairs)
df_pairs["pair_type"] = np.where(
    (df_pairs["g1"] == "high") & (df_pairs["g2"] == "high"),
    "HH",
    np.where(
        (df_pairs["g1"] == "low") & (df_pairs["g2"] == "low"),
        "LL",
        "HL"
    )
)

# Optional: remove self-pairs if present
df_pairs = df_pairs[df_pairs["iid1"] != df_pairs["iid2"]]

# --- 4) Extract distributions
hh = df_pairs.loc[df_pairs["pair_type"] == "HH", IBD_METRIC].to_numpy()
hl = df_pairs.loc[df_pairs["pair_type"] == "HL", IBD_METRIC].to_numpy()
ll = df_pairs.loc[df_pairs["pair_type"] == "LL", IBD_METRIC].to_numpy()

# --- 5) Quick descriptive summary
summary = (
    df_pairs.groupby("pair_type")[IBD_METRIC]
    .agg(n="count", mean="mean", median="median", std="std")
    .sort_index()
)
print("\n=== Descriptives by pair type ===")
print(summary)

# --- 6) Tests
# 6a) One-sided Mann–Whitney: HH > HL (your main question)
if len(hh) > 0 and len(hl) > 0:
    u_hh_hl = mannwhitneyu(hh, hl, alternative="greater")
    print("\n=== Mann–Whitney U: HH > HL ===")
    print(f"U={u_hh_hl.statistic:.3f}, p={u_hh_hl.pvalue:.3g}")
else:
    print("\n[Skip] Not enough data for HH vs HL Mann–Whitney.")

# 6b) One-sided Mann–Whitney: HH > LL (often also interesting)
if len(hh) > 0 and len(ll) > 0:
    u_hh_ll = mannwhitneyu(hh, ll, alternative="greater")
    print("\n=== Mann–Whitney U: HH > LL ===")
    print(f"U={u_hh_ll.statistic:.3f}, p={u_hh_ll.pvalue:.3g}")
else:
    print("\n[Skip] Not enough data for HH vs LL Mann–Whitney.")

# 6c) Omnibus (3 groups): Kruskal–Wallis (non-parametric ANOVA)
groups_for_kw = [arr for arr in (hh, hl, ll) if len(arr) > 0]
labels_for_kw = [lab for lab, arr in zip(("HH", "HL", "LL"), (hh, hl, ll)) if len(arr) > 0]
if len(groups_for_kw) >= 2:
    kw = kruskal(*groups_for_kw)
    print("\n=== Kruskal–Wallis omnibus ===")
    print(f"Groups={labels_for_kw}, H={kw.statistic:.3f}, p={kw.pvalue:.3g}")
else:
    print("\n[Skip] Not enough groups for Kruskal–Wallis.")

# --- 7) Effect size (Cliff's delta) for HH vs HL (robust, interpretable)
def cliffs_delta(x, y):
    """
    Cliff's delta: P(x>y) - P(x<y)
    Range [-1, 1]. Positive means x tends to be larger than y.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    # broadcast compare (can be heavy for huge arrays; for big data consider sampling)
    gt = (x[:, None] > y[None, :]).sum()
    lt = (x[:, None] < y[None, :]).sum()
    return (gt - lt) / (len(x) * len(y))

if len(hh) > 0 and len(hl) > 0:
    delta = cliffs_delta(hh, hl)
    print("\n=== Effect size: Cliff's delta (HH vs HL) ===")
    print(f"delta={delta:.3f}  (positive => HH tends higher than HL)")

# --- 8) If you want the filtered pair table for export/debug
# df_pairs.to_csv("pairs_with_cts_groups.csv", index=False)


In [ ]:
df_ibds2 = pd.read_csv(
    "/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_EIAonly_12cM_fracgp0.7.csv",
    sep=","
)

# all iids in the TSV
iids_in_iid = set(df_iid["iid"])

# all iids that actually appear in df_ibds2
iids_in_ibds2 = set(df_ibds2["iid1"]).union(df_ibds2["iid2"])

# iids present in df_iid but missing from df_ibds2
missing_iids = sorted(iids_in_iid - iids_in_ibds2)

# as a list
missing_iids


In [ ]:

df_ibds1 = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12.csv",  sep=",")
df_ibds2 = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_12cM_fracgp0.7.csv",  sep=",")


df_ibds2 = df_ibds2[(df_ibds2["iid1"].str[:3]=="KNC")|(df_ibds2["iid2"].str[:3]=="KNC")].sort_values(by="max_IBD", ascending=False)

In [ ]:
df_ibds2[df_ibds2["iid1"].str[:3]!=df_ibds2["iid2"].str[:3]].sort_values(by="sum_IBD>20", ascending=False)[:50]

### Plot all IBD for one pair
Goal: Zoom in later

In [ ]:
path_ibd = "/mnt/archgen/users/hringbauer/output/ibd/KNC.v12/ch_all.tsv"
df_ibd = pd.read_csv(path_ibd, sep="\t")
plot_karyo_from_ibd_df(df_ibd, iids=["KNC096", "KNC033"], min_cm=12, savepath="", 
                       title="KNC096 - KNC151") 

In [ ]:
import pandas as pd

# === 1. Load the TSV ===
df = pd.read_csv("/mnt/archgen/users/hringbauer/output/ibd/KNC.v12/ch_all.tsv", sep="\t")

# === 2. Define individual IDs ===
bro1 = "KNC033"
bro2 = "KNC151"
cand = "KNC096"

# === 3. Keep only rows involving these 3 ===
df = df[((df['iid1'].isin([bro1, bro2, cand])) & 
         (df['iid2'].isin([bro1, bro2, cand])))].copy()

# === 4. Helper: get subset for a specific pair (iid order doesn’t matter) ===
def get_pair(df, id1, id2):
    return df[((df['iid1'] == id1) & (df['iid2'] == id2)) |
              ((df['iid1'] == id2) & (df['iid2'] == id1))].copy()

bro1_cand = get_pair(df, bro1, cand)
bro2_cand = get_pair(df, bro2, cand)

# === 5. Compute total shared bases for each pair ===
def total_bp(d):
    return (d['EndBP'] - d['StartBP']).sum()

total1 = total_bp(bro1_cand)
total2 = total_bp(bro2_cand)

# === 6. Compute overlap between candidate–bro1 and candidate–bro2 ===
# We'll do it chromosome by chromosome
overlap_bp = 0

for chrom in sorted(set(bro1_cand['ch']) & set(bro2_cand['ch'])):
    a = bro1_cand[bro1_cand['ch'] == chrom][['StartBP','EndBP']].values
    b = bro2_cand[bro2_cand['ch'] == chrom][['StartBP','EndBP']].values
    for s1,e1 in a:
        for s2,e2 in b:
            start = max(s1,s2)
            end   = min(e1,e2)
            if end > start:
                overlap_bp += (end - start)

# === 7. Compute overlap fraction ===
overlap_fraction = overlap_bp / ((total1 + total2) / 2)

print(f"Total shared (bro1-cand): {total1/1e6:.1f} Mb")
print(f"Total shared (bro2-cand): {total2/1e6:.1f} Mb")
print(f"Total overlap between both: {overlap_bp/1e6:.1f} Mb")
print(f"Overlap fraction: {overlap_fraction:.3f}")

# === 8. Simple interpretation heuristic ===
if overlap_fraction >= 0.35:
    rel = "likely UNCLE–NEPHEW (shared tracts overlap strongly)"
elif overlap_fraction <= 0.25:
    rel = "likely HALF-BROTHER (little overlap)"
else:
    rel = "ambiguous—intermediate overlap"
print("Interpretation:", rel)


## Highlight Single Sample

In [ ]:
import matplotlib.pyplot as plt
from adjustText import adjust_text
import numpy as np

def plot_scatter_ibd_pairs_adjust(df_background, df_targets, pairs=[], min_cm=12, s=40,
                                  xlim=[-5,3600], ylim=[0,50], 
                                  c_back="gray", c_target="maroon", savepath="",dpi=300):
    """
    Scatter plot of sum and number of IBD segments with smart label placement.
    
    df_background: background DataFrame
    df_targets: DataFrame to search for highlighted pairs
    pairs: list of tuples, each tuple is a pair of iids to highlight (search in df_targets)
    """
    
    plt.figure(figsize=(8,8),dpi=dpi)
    ax = plt.gca()
    
    # Plot background points
    ax.scatter(df_background[f"sum_IBD>{min_cm}"], df_background[f"n_IBD>{min_cm}"], 
               s=s, ec="k", linewidth=0.3, color=c_back, zorder=0,alpha=0.7)
    
    texts = []  # store text objects for adjustText
    
    # Highlight pairs
    for pair in pairs:
        iid1, iid2 = pair
        idx = ((df_targets["iid1"] == iid1) & (df_targets["iid2"] == iid2)) | \
              ((df_targets["iid1"] == iid2) & (df_targets["iid2"] == iid1))
        dft = df_targets[idx]
        if len(dft) > 0:
            # Plot all matching points
            ax.scatter(dft[f"sum_IBD>{min_cm}"], dft[f"n_IBD>{min_cm}"], 
                       s=s*2, ec="k", linewidth=0.8, color=c_target, zorder=10)
            
            # Create text objects for all points
            for x, y in zip(dft[f"sum_IBD>{min_cm}"], dft[f"n_IBD>{min_cm}"]):
                txt = ax.text(x, y, f"{iid1}-{iid2}", fontsize=14, color=c_target,
                              bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
                texts.append(txt)
    
    # Adjust text positions automatically to avoid overlap
    adjust_text(texts, force_text=0.75, expand_points=(1.5, 1.5), expand_text=(1.5, 1.5), only_move={'points':'y', 'texts':'xy'}, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
    
    ax.set_xlabel(f"Sum IBD >{min_cm}cM [cM]", fontsize=14)
    ax.set_ylabel(f"n IBD >{min_cm}cM", fontsize=14)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    
    if len(savepath) > 0:
        plt.savefig(savepath, bbox_inches='tight', pad_inches=0, dpi=400)
        print(f"Saved to {savepath}")
    
    plt.show()


In [ ]:
#pairs = [("KNC099", "KNC139"), ("KNC099", "KNC172"), ("KNC099", "KNC076"), ("KNC139", "KNC172"), ("KNC139", "KNC076")]
pairs = [ ("KNC107", "KNC044")]
#pairs = [ ("KNC078", "KNC011"), ("KNC078", "KNC086"), ("KNC078", "KNC082"), ("KNC078", "KNC033"), ("KNC078", "KNC151"), ("KNC078", "KNC035")]

df_ibds1 = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12.csv",  sep=",")
df_ibds2 = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_12cM_fracgp0.7.csv",  sep=",")
print(df_ibds1.columns.tolist())
print(df_ibds2.columns.tolist())

plot_scatter_ibd_pairs_adjust(df_background=df_ibds2, 
                                  df_targets=df_ibds1, 
                                  pairs=pairs,
                                  c_back='silver', 
                                  c_target="red")
